In [1]:
import pandas as pd

FAMILLES_CIBLES = [
    'DDoS',
    'DoS',
    'Mirai',
    'Recon',
    'Spoofing',
    'Injection'
]

LABEL_BENIN = 'BenignTraffic'

MAPPING_EXPLICITE = {
    'DNS_Spoofing': 'Spoofing',
    'MITM-ArpSpoofing': 'Spoofing',

    'SqlInjection': 'Injection',
    'CommandInjection': 'Injection',
    'XSS': 'Injection',
    'Uploading_Attack': 'Injection'
}

def mapper_famille(label):
    if label == LABEL_BENIN:
        return LABEL_BENIN

    if label in MAPPING_EXPLICITE:
        return MAPPING_EXPLICITE[label]

    for famille in FAMILLES_CIBLES:
        if label.startswith(famille):
            return famille

    return None

df = pd.read_csv("../data/cache_filtre.csv")

print("Avant :", df["label"].nunique())

df["label"] = df["label"].apply(mapper_famille)

print("Après :", df["label"].nunique())

print(df["label"].value_counts())

print("Non mappés :", df["label"].isna().sum())

df.to_csv("../data/cache_filtre.csv", index=False)

Avant : 7
Après : 7
label
DDoS             150000
DoS              150000
Mirai            150000
BenignTraffic    150000
Spoofing         150000
Recon            147124
Injection          7363
Name: count, dtype: int64
Non mappés : 0


In [2]:
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

df = pd.read_csv("../data/cache_filtre.csv")
le = LabelEncoder()
df['label_num'] = le.fit_transform(df['label'])
X = df.drop(columns=['label', 'label_num'])
y = df['label_num'].values

# Split d'abord, puis fit sur train seulement
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SMOTE Balancing — sur les données SCALÉES, pas les brutes
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

# Shaping for RNN (Batch, Time_Steps, Features) — sur les données SCALÉES
X_train_rnn = X_train_res.reshape(X_train_res.shape[0], 1, X_train_res.shape[1])
X_test_rnn = X_test_scaled.reshape(X_test_scaled.shape[0], 1, X_test_scaled.shape[1])

joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le, '../models/label_encoder.pkl')

['../models/label_encoder.pkl']